In [ ]:
## 2026.08.20 CODEX GIST twin of CODEX_hcc/HCC_train_validate_cv_UNIlabel_all.ipynb
## Cross-dataset training on 550 annotated GIST TMA cores (s1167)
##
## CLI equivalent:
##   conda run --no-capture-output -n SeededNTM python -u \
##     code/CODEX_gist/GIST_train_validate_cv_UNIlabel.py --mode cross-dataset \
##     --use-spatial-context --spatial-k 8 --spatial-mode mean \
##     --pooled-save-result result_all_spatial
##
## Hierarchy (three-head):
##   L2  = final_CT          (fine / celltype_level2)
##   L12 = final_sublineage  (intermediate / celltype_level1)
##   L1  = final_lineage     (coarse / celltype_level0)
##
## Preprocess CSVs first:
##   conda run -n SeededNTM python code/CODEX_gist/match_codex_cells_with_pixel.py
## Env: conda activate SeededNTM


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("NCRT_CUDA_DEVICE", "0")
import sys as _sys
_pkg_dir = "/home/lingyu/ssd2/Python/Hist2Pheno/code/Hist2Pheno_pkg"
if _pkg_dir not in _sys.path:
    _sys.path.insert(0, _pkg_dir)
from runtime_env import configure_notebook_runtime
_rt_env = configure_notebook_runtime()
NCRT_PHYSICAL_GPU = _rt_env["physical_gpu"]


In [ ]:
import sys
import importlib
import pandas as pd
from pathlib import Path

path = "/home/lingyu/ssd2/Python/"
for _d in (
    f"{path}Hist2Pheno/code/Hist2Pheno_pkg",
    f"{path}Hist2Pheno/code/CODEX_gist",
):
    if _d not in sys.path:
        sys.path.insert(0, _d)

import base
import plot
import model as model_pkg
importlib.reload(base)
importlib.reload(plot)
importlib.reload(model_pkg)

import uni_label_cv_helpers as uni_nb
importlib.reload(uni_nb)

import GIST_train_validate_cv_UNIlabel as gist_cli
importlib.reload(gist_cli)

print("Imports OK")
print(f"CUDA_VISIBLE_DEVICES={os.environ.get('CUDA_VISIBLE_DEVICES')} (physical GPU {NCRT_PHYSICAL_GPU})")


In [ ]:
## CUDA/CPU sanity check + reproducibility
import torch
import random
import numpy as np

print(sys.executable)
print(f"torch {torch.__version__} | built with CUDA: {torch.version.cuda} | cuda.is_available: {torch.cuda.is_available()}")
gist_cli.setup_cuda(os.environ.get("NCRT_CUDA_DEVICE", "0"))
device = gist_cli.setup_device(allow_cpu=False)
SEED = 42
gist_cli.setup_seed(SEED)
print(f"device: {device}")


## Cross-dataset config

Pool `s1167/{ACQUISITION_ID}/{ACQUISITION_ID}_matched_features.h5ad` for the **550 annotated** GIST cores.
Pancreas TMA / PDAC is a separate cohort (`code/CODEX_pdac`). All 550 GIST cores are annotated.


In [ ]:
from uni_label_cv_helpers import (
    RESULT_FIG,
    make_pooled_result_fig,
    make_pooled_stardist_result_fig,
    stardist_all_label_h5ad_path,
    )

S1167_ROOT = Path(f"{path}Hist2Pheno/data/CODEX/HCC/Michael_data_transfer/s1167")
DATA_ROOT = S1167_ROOT
CASES_ROOT = S1167_ROOT
save_result = "result_all_spatial"

QUICK_VALIDATE = False
VALIDATION_SUBSET = [
    "Charvill-94_c013_v001_r001_reg002",
    "Charvill-94_c011_v001_r001_reg002",
    "Charvill-94_c009_v001_r001_reg001",
]

all_samples = gist_cli.discover_gist_samples(CASES_ROOT, require_h5ad=False, require_cells_csv=True)
_acq_map = gist_cli.load_he_acq_map(CASES_ROOT)
_target = VALIDATION_SUBSET if QUICK_VALIDATE else all_samples
for _s in _target:
    _h5 = CASES_ROOT / _s / f"{_s}_matched_features.h5ad"
    if _h5.is_file():
        continue
    print(f"Building HE h5ad for {_s} ...", flush=True)
    _ctx = gist_cli.RunContext(
        sample=_s,
        cases_root=CASES_ROOT,
        python_root=Path(path),
        therapy_data=_s,
        therapy_model=gist_cli.DEFAULT_THERAPY_MODEL,
        save_result="result",
        device=device,
        seed=SEED,
        match_tolerance=1.0,
        column_rename=dict(gist_cli.HCC_COLUMN_RENAME),
        force_rebuild_h5ad=False,
        input_dim=None,
        hidden_dims=(1024, 512, 256),
        cv_k=5,
        stratify_target="joint",
        patience=10,
        max_epochs=50,
        train_batch_size=4096,
        resume_from_checkpoints=True,
        ablation_tag="D_emph_L2_spatial",
        hce_w1=1.0, hce_w2=2.0, hce_w12=1.0, hce_w_l12head=1.0, hce_w_l3=1.0, hce_w_l4=1.0,
        build_stardist_h5ad=False,
        acq_id=_acq_map[_s],
        use_spatial_context=True,
        spatial_k=8,
        spatial_mode="mean",
    )
    gist_cli.step_he_h5ad(_ctx)

POOLED_SAMPLES = gist_cli.discover_gist_samples(
    CASES_ROOT, sample=None, require_h5ad=True, require_cells_csv=True,
)
if QUICK_VALIDATE:
    POOLED_SAMPLES = [s for s in VALIDATION_SUBSET if s in POOLED_SAMPLES]
print(f"Cross-dataset: {len(POOLED_SAMPLES)} datasets (GIST three-head)")

pooled_result_fig, pooled_result_dir = make_pooled_result_fig(DATA_ROOT, save_result)
FIG = RESULT_FIG
print(f"Results → {pooled_result_dir}")

POOLED_CV_K = 2 if QUICK_VALIDATE else 5
POOLED_MAX_EPOCHS = 1 if QUICK_VALIDATE else 50
POOLED_PATIENCE = 1 if QUICK_VALIDATE else 10
POOLED_TRAIN_GROUP_FRAC = 0.7
POOLED_TRAIN_BATCH_SIZE = 256 if QUICK_VALIDATE else 4096
POOLED_USE_SPATIAL = True
POOLED_SPATIAL_K = 8
POOLED_SPATIAL_MODE = "mean"
if POOLED_USE_SPATIAL and not QUICK_VALIDATE:
    POOLED_ABLATION_TAG = "D_emph_L2_spatial_bs4096"
elif QUICK_VALIDATE:
    POOLED_ABLATION_TAG = "D_emph_L2_smoke"
else:
    POOLED_ABLATION_TAG = "D_emph_L2"
POOLED_HIDDEN_DIMS = (1024, 512, 256)
POOLED_HCE = dict(hce_w1=1.0, hce_w2=2.0, hce_w12=1.0, hce_w_l12head=1.0, hce_w_l3=1.0, hce_w_l4=1.0)
POOLED_VAL_SELECTION = gist_cli.DEFAULT_VAL_SELECTION_METRIC
POOLED_CV_SELECTION = gist_cli.DEFAULT_CV_SELECTION_METRIC

pooled_ctx = gist_cli.PooledRunContext(
    data_root=DATA_ROOT,
    cases_root=CASES_ROOT,
    python_root=Path(path),
    samples=POOLED_SAMPLES,
    device=device,
    seed=SEED,
    save_result=save_result,
    input_dim=None,
    hidden_dims=POOLED_HIDDEN_DIMS,
    cv_k=POOLED_CV_K,
    train_group_frac=POOLED_TRAIN_GROUP_FRAC,
    patience=POOLED_PATIENCE,
    max_epochs=POOLED_MAX_EPOCHS,
    train_batch_size=POOLED_TRAIN_BATCH_SIZE,
    resume_from_checkpoints=not QUICK_VALIDATE,
    ablation_tag=POOLED_ABLATION_TAG,
    val_selection_metric=POOLED_VAL_SELECTION,
    cv_selection_metric=POOLED_CV_SELECTION,
    use_spatial_context=POOLED_USE_SPATIAL,
    spatial_k=POOLED_SPATIAL_K,
    spatial_mode=POOLED_SPATIAL_MODE,
    **POOLED_HCE,
)
print(f"Selection: val={POOLED_VAL_SELECTION}, cv={POOLED_CV_SELECTION}")
print(f"Ablation tag: {POOLED_ABLATION_TAG}, batch_size={POOLED_TRAIN_BATCH_SIZE}")
if POOLED_USE_SPATIAL:
    print(f"Spatial context: k={POOLED_SPATIAL_K}, mode={POOLED_SPATIAL_MODE!r}")


## §1 Load pooled h5ad and prepare CV arrays


In [ ]:
gist_cli.step_pooled_prepare(pooled_ctx)


## §2 Group K-fold training


In [ ]:
gist_cli.step_pooled_train(pooled_ctx)
print('BEST_MLP_CHECKPOINT:', pooled_ctx.g.get('BEST_MLP_CHECKPOINT'))


## §3 Internal validation (OOF) — L2 / L12 / L1


In [ ]:
gist_cli.step_pooled_he_validate(pooled_ctx)


## §4 StarDist predict (matched, with GT AUROC)

Requires `{acq}_matched_features_stardist.h5ad` from `transer_embedding_label_h5ad.py`.
Outputs under `s1167/{save_result}/stardist/{ACQUISITION_ID}/`.


In [ ]:
gist_cli.step_pooled_stardist(pooled_ctx, levels=["l2"])
gist_cli.step_pooled_stardist(pooled_ctx, levels=["l1"])
gist_cli.step_pooled_stardist(pooled_ctx, levels=["l12"])


## §5 StarDist predict on **all nuclei** of the **550 annotated** cores

Input: `s1167/{acq}/{acq}_all_features_stardist.h5ad`
Output: `s1167/{save_result}/stardist/{acq}/{acq}_all_features_stardist_label.h5ad`


In [ ]:
if "BEST_MLP_CHECKPOINT" not in pooled_ctx.g:
    raise RuntimeError("Run §2 (step_pooled_train) first.")
gist_cli.step_pooled_stardist_all(pooled_ctx)
for sample in POOLED_SAMPLES[:5]:
    print(" ", stardist_all_label_h5ad_path(DATA_ROOT, sample, save_result))
if len(POOLED_SAMPLES) > 5:
    print(f"  ... +{len(POOLED_SAMPLES) - 5} more samples")


In [ ]:
import anndata as ad

# Inspect the first GIST sample with a written StarDist-all label h5ad.
_demo = next(
    (
        s for s in POOLED_SAMPLES
        if stardist_all_label_h5ad_path(DATA_ROOT, s, save_result).is_file()
    ),
    None,
)
if _demo is None:
    print("No StarDist-all label h5ad yet — run the cell above (and build missing *_all_features_stardist.h5ad).")
else:
    _h5ad = stardist_all_label_h5ad_path(DATA_ROOT, _demo, save_result)
    print(f"Demo GIST: {_demo}\n  {_h5ad}")
    _a = ad.read_h5ad(_h5ad, backed="r")
    try:
        prob_cols = [c for c in _a.obs.columns if "_prob_" in c]
        print(f"  n_obs={_a.n_obs:,}  prob columns={len(prob_cols)}")
        print(f"  heads in uns: {list(_a.uns.get('pred_prob_class_names', {}).keys())}")
        print(f"  sample obs cols: {list(_a.obs.columns[:8])} ...")
    finally:
        _a.file.close()


### Plot cell type


In [ ]:
## Predicted spatial maps: GIST StarDist-all × L2 / L12 / L1 (pred-only).
## JPGs: stardist/{acq}/{acq}_stardist_pred_{l2,l12,l1}.jpg
## Overview defaults to 2 cores per coverslip (550×3 is too large).
import importlib
import uni_label_cv_helpers as uni_nb
import s1167_plot
importlib.reload(uni_nb)
importlib.reload(s1167_plot)

if "POOLED_SAMPLES" not in globals() or not POOLED_SAMPLES:
    POOLED_SAMPLES = gist_cli.discover_gist_samples(
        CASES_ROOT, require_h5ad=True, require_cells_csv=True,
    )

print(f"Plotting GIST StarDist-all × {s1167_plot.GIST_STARDIST_MACRO_AUROC_TIERS}")
gist_spatial = s1167_plot.plot_gist_stardist_spatial_maps(
    DATA_ROOT, POOLED_SAMPLES, save_result, pan_organ=gist_cli.PAN_ORGAN,
)


In [ ]:
s1167_plot.plot_gist_stardist_spatial_overview(
    gist_spatial, DATA_ROOT, save_result,
    pan_organ=gist_cli.PAN_ORGAN, point_size=0.5,
)


## Notes

- GIST has **no** CNiche/TNiche (L3/L4); only L2 / L12 / L1.
- All **550** GIST TMA cores are annotated (c009 160 + c011 197 + c013 193); there is no Incomplete_Cases track.
- §5 writes `stardist/` all-nuclei labels under `s1167/{save_result}/stardist/{ACQUISITION_ID}/`.
- Pancreas TMA / PDAC pipeline: `code/CODEX_pdac`.
- Batch CLI: `GIST_train_validate_cv_UNIlabel.py --mode cross-dataset`
